In [1]:
import torch
import numpy as np
from tqdm import tqdm
import random

import os
os.chdir("..")

from src.models.exogenous_transformer import ExogenousTransformer
from src.shap import get_owen_masks
from src.data import load_data

## Define the settings

To set up a SHAPformer model, we need to provide a list of features. For each feature, an entry in `CATEGORIES` indicates whether it is continuous (indicated with `None`) or categorical (by providing the number of categories).

In [2]:
FEATURES = [
    "load",
    "hourofday",
    "dayofweek",
    "month",
    "holiday",
    "temperature",
    "precipitation"
]

CATEGORIES = [None, 24, 7, 12, 2, None, None]

EPOCHS = 20
STEPS = 1000
DEVICE = "cuda"

## Create the model

Next, we initialize a SHAPformer model. For demonstration purposes, we choose a small model with 1 layer, 1 attention head and a hidden dimension of 32.

In [3]:
model = ExogenousTransformer(
    encoder_feature_names=FEATURES,
    encoder_categories=CATEGORIES,
    decoder_feature_names=FEATURES[1:],
    decoder_categories=CATEGORIES[1:],
    d_model=32,
    n_layers=1,
    n_heads=1
)
model.to(DEVICE)
print(model)

ExogenousTransformer(
  (encoder_embedding): FeatureEmbedding(
    (embeddings): ParameterDict(
        (load): Object of type: LinearEmbedding
        (hourofday): Object of type: CategoricalEmbedding
        (dayofweek): Object of type: CategoricalEmbedding
        (month): Object of type: CategoricalEmbedding
        (holiday): Object of type: CategoricalEmbedding
        (temperature): Object of type: LinearEmbedding
        (precipitation): Object of type: LinearEmbedding
      (load): LinearEmbedding(
        (embedding): Linear(in_features=1, out_features=32, bias=True)
      )
      (hourofday): CategoricalEmbedding(
        (embedding): Embedding(24, 32)
      )
      (dayofweek): CategoricalEmbedding(
        (embedding): Embedding(7, 32)
      )
      (month): CategoricalEmbedding(
        (embedding): Embedding(12, 32)
      )
      (holiday): CategoricalEmbedding(
        (embedding): Embedding(2, 32)
      )
      (temperature): LinearEmbedding(
        (embedding): Linea

## Load the training data

In [4]:
train_data = load_data("data/transnet/train.json", device=DEVICE)

loss_function = torch.nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.0001)

n_features = len(FEATURES)

Now let's look at the first sample. It is a dictionary with three entries:

- `x_enc` for the encoder input, i.e. the information about the past time steps.
- `x_dec` for the decoder input, i.e. the information about the future time steps.
- `y` for the target.

`x_enc`and `x_dec` contain one time series per feature, with 168 entries each.
Note that the "load" feature only exists for the past.

In [5]:
sample = train_data[0]
for key in ["x_enc", "x_dec"]:
    for feature in sample[key]:
        print(key, feature, sample[key][feature].shape)
print("y", sample["y"].shape)

x_enc load torch.Size([168])
x_enc hourofday torch.Size([168])
x_enc dayofweek torch.Size([168])
x_enc month torch.Size([168])
x_enc holiday torch.Size([168])
x_enc temperature torch.Size([168])
x_enc precipitation torch.Size([168])
x_dec hourofday torch.Size([168])
x_dec dayofweek torch.Size([168])
x_dec month torch.Size([168])
x_dec holiday torch.Size([168])
x_dec temperature torch.Size([168])
x_dec precipitation torch.Size([168])
y torch.Size([168])


## Generate all feature subset masks for masked training

For the masked training, we need feature masks and time point masks.
The function `get_owen_masks` generates all possible masks for the given number of features.
A mask key consists of `n_features - 1 + 7` binary entries, for the `n_features - 1` exogenous features and seven past days.
An entry of 0 means that the feature group is masked, and an entry of 1 that it is accessible to the model.
The `masks` dictionary maps each mask key to a tuple with a feature mask for the masked feature attention mechanism and a temporal mask for the self-attention and cross-attention of the model.

In [6]:
masks = get_owen_masks(n_features, device=DEVICE)
mask_keys = list(masks)

print(mask_keys[:5])

[(False, False, False, False, False, False, False, False, False, False, False, False, False), (False, False, False, False, False, False, False, False, False, False, False, False, True), (False, False, False, False, False, False, False, False, False, False, False, True, False), (False, False, False, False, False, False, False, False, False, False, False, True, True), (False, False, False, False, False, False, False, False, False, False, True, False, False)]


## Training loop

The model gets trained on random samples from the training set, which are masked by randomly selected masks.
Note that the decoder feature mask `mask_dec` is smaller than the encoder feature mask `mask_enc`, because the load feature is only available in the encoder input.

This exemplary training loop uses a batch size of 1 and runs for a fixed number of epochs. For more accurate forecasting, consider using a larger batch size and early stopping on the validation set.

In [7]:
indices = list(range(len(train_data)))

for epoch in range(EPOCHS):
    np.random.shuffle(indices)
    total_loss = 0
    for i in tqdm(indices[:STEPS]):
        optimizer.zero_grad()

        sample = train_data[i]
        x_enc = sample["x_enc"]
        x_dec = sample["x_dec"]
        y = sample["y"]

        x_enc = {k: v[None, :] for k, v in x_enc.items()}
        x_dec = {k: v[None, :] for k, v in x_dec.items()}
        y = y[None, :]

        #ft_mask, attn_mask = get_full_masks(n_features, device=DEVICE)
        ft_mask, attn_mask = masks[random.choice(mask_keys)]

        output = model.forward(x_enc, x_dec, mask_enc=ft_mask[None, :], mask_dec=ft_mask[None, 1:], attn_mask=attn_mask[None, :, :])

        loss = loss_function(output, y)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch + 1}/{EPOCHS}, Loss: {total_loss / STEPS}")

100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1000/1000 [00:16<00:00, 62.17it/s]


Epoch 1/20, Loss: 0.9198443663716316


100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1000/1000 [00:15<00:00, 64.53it/s]


Epoch 2/20, Loss: 0.7402602797001601


100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1000/1000 [00:15<00:00, 66.22it/s]


...

Epoch 18/20, Loss: 0.42418099673464893


100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1000/1000 [00:15<00:00, 63.20it/s]


Epoch 19/20, Loss: 0.4223062208369374


100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1000/1000 [00:15<00:00, 64.10it/s]

Epoch 20/20, Loss: 0.40202846443653106


## Save the trained model

Finally, we save the trained model. See the notebook evaluation.ipynb for how to use the model to generate forecasts and explain them.

In [8]:
torch.save(model, "exogenous_transformer.pt")